In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path(
    "../data/processed/cleaned_grading_dataset.xlsx"
)

df = pd.read_excel(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

print(df.head())

Dataset shape: (1055, 116)
Columns: 116
                      Tissue_ID    Eye                Image_ID Disease_Label  \
0  12346 R (year/month unknown)  Right                     NaN        Normal   
1           NEB-2020-09-12654 R  Right  NEB-2020-09-12654 R_SP        Normal   
2           NEB-2020-09-12655 L   Left  NEB-2020-09-12655 L_SP         Folds   
3           NEB-2020-09-12656 R  Right  NEB-2020-09-12656 R_SP        Normal   
4           NEB-2020-09-12657 L   Left  NEB-2020-09-12657 L_SP         Folds   

  Disease_Severity  ECD_CD_cells_per_mm2    CV  HEX_percent     SD  \
0              NaN                   NaN   NaN          NaN    NaN   
1              NaN                2132.0  33.0         58.0  153.0   
2         Moderate                2212.0  33.0         47.0  147.0   
3              NaN                2538.0  35.0         53.0  137.0   
4             Mild                2584.0  29.0         69.0  112.0   

   Total_Cell_Count  ...  Epi_Has_Exposure Epi_Exposure_Se

In [3]:
TARGET = "Tissue_Grade_Clean"

assert TARGET in df.columns, f"Missing target column: {TARGET}"

print("Target value counts, including missing values:")
display(df[TARGET].value_counts(dropna=False))

print("\nTarget percentages among labelled records:")
display(
    (df[TARGET].value_counts(normalize=True) * 100).round(2)
)

print("\nMissing target values:", df[TARGET].isna().sum())
print("Labelled records:", df[TARGET].notna().sum())

Target value counts, including missing values:


Tissue_Grade_Clean
A+     545
A      319
B      172
NaN     19
Name: count, dtype: int64


Target percentages among labelled records:


Tissue_Grade_Clean
A+    52.61
A     30.79
B     16.60
Name: proportion, dtype: float64


Missing target values: 19
Labelled records: 1036


In [4]:
VALID_GRADES = ["A+", "A", "B"]

labelled_df = df[df[TARGET].isin(VALID_GRADES)].copy()

print("Original rows:", len(df))
print("Rows with valid specialist grades:", len(labelled_df))
print("Excluded rows with missing or unexpected grades:",
      len(df) - len(labelled_df))

display(labelled_df[TARGET].value_counts().rename("Count").to_frame())

Original rows: 1055
Rows with valid specialist grades: 1036
Excluded rows with missing or unexpected grades: 19


,Count
Tissue_Grade_Clean,
A+,545
A,319
B,172


In [5]:
identifier_candidates = [
    "Tissue_ID",
    "Eye",
    "Image_ID",
    "Donor_ID",
    "Donor_ID_Clean",
]

identifier_summary = []

for col in identifier_candidates:
    if col in labelled_df.columns:
        identifier_summary.append({
            "Column": col,
            "Non-null": labelled_df[col].notna().sum(),
            "Unique values": labelled_df[col].nunique(dropna=True),
            "Missing": labelled_df[col].isna().sum(),
        })

display(pd.DataFrame(identifier_summary))

,Column,Non-null,Unique values,Missing
0,Tissue_ID,1036,1036,0
1,Eye,1036,2,0
2,Image_ID,999,999,37


In [6]:
potential_leakage_columns = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes",
    "Additional_Findings",
]

leakage_columns_present = [
    col for col in potential_leakage_columns
    if col in labelled_df.columns
]

print("Potential leakage/context columns found:")
for col in leakage_columns_present:
    print("-", col)

print("\nColumns not found in this dataset:")
for col in potential_leakage_columns:
    if col not in labelled_df.columns:
        print("-", col)

Potential leakage/context columns found:
- Overall_Donor_Quality
- Transplant_Suitability
- Suitability_Reason
- Tissue_Outcome
- Recipient_Age
- Recipient_Eye
- Recipient_Indication
- Request_Indication
- Primary_Procedure
- Secondary_Procedures
- Surgery_Date
- Requested_Grade
- Planned_Procedure
- Grade_Request_Met
- Review_Notes
- Additional_Findings

Columns not found in this dataset:


In [7]:
numeric_candidates = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Donor_Age",
]

clinical_candidates = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
    "Epi_Has_Exposure",
    "Epi_Exposure_Severity",
    "Epi_Exposure_Location",
    "Epi_Has_Defect",
    "Epi_Has_Haze",
    "Epi_Haze_Severity",
    "Epi_Has_Arcus",
]

numeric_present = [
    col for col in numeric_candidates
    if col in labelled_df.columns
]

clinical_present = [
    col for col in clinical_candidates
    if col in labelled_df.columns
]

print("Numeric candidates:", len(numeric_present))
print(numeric_present)

print("\nClinical candidates:", len(clinical_present))
print(clinical_present)

print("\nMissing candidate columns:")
print("Numeric:", sorted(set(numeric_candidates) - set(numeric_present)))
print("Clinical:", sorted(set(clinical_candidates) - set(clinical_present)))

Numeric candidates: 7
['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2', 'Donor_Age']

Clinical candidates: 24
['Stroma_Is_Clear', 'Stroma_Is_Compact', 'Stroma_Has_Arcus', 'Stroma_Has_Edema', 'Stroma_Has_Scar', 'Stroma_Arcus_Severity', 'Stroma_Scar_Type', 'Descemet_Has_Folds', 'Descemet_Fold_Severity', 'Descemet_Fold_Relative_Number', 'Endo_Has_Stress_Lines', 'Endo_Stress_Line_Location', 'Endo_Stress_Line_Amount', 'Endo_Has_Cell_Dropout', 'Endo_Cell_Dropout_Severity', 'Endo_Has_RBC', 'Endo_RBC_Severity', 'Epi_Has_Exposure', 'Epi_Exposure_Severity', 'Epi_Exposure_Location', 'Epi_Has_Defect', 'Epi_Has_Haze', 'Epi_Haze_Severity', 'Epi_Has_Arcus']

Missing candidate columns:
Numeric: []
Clinical: []


In [8]:
identifier_like = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "donor", "tissue_id", "image_id", "serial",
        "accession", "specimen", "record_id"
    ])
]

print("Identifier-like columns:")
for col in identifier_like:
    print(f"\n{col}")
    print("  dtype:", df[col].dtype)
    print("  non-null:", df[col].notna().sum())
    print("  unique:", df[col].nunique(dropna=True))
    print("  examples:", df[col].dropna().astype(str).head(5).tolist())

Identifier-like columns:

Tissue_ID
  dtype: str
  non-null: 1055
  unique: 1055
  examples: ['12346 R (year/month unknown)', 'NEB-2020-09-12654 R', 'NEB-2020-09-12655 L', 'NEB-2020-09-12656 R', 'NEB-2020-09-12657 L']

Image_ID
  dtype: str
  non-null: 1013
  unique: 1013
  examples: ['NEB-2020-09-12654 R_SP', 'NEB-2020-09-12655 L_SP', 'NEB-2020-09-12656 R_SP', 'NEB-2020-09-12657 L_SP', 'NEB-2020-09-12658 R_SP']

Overall_Donor_Quality
  dtype: str
  non-null: 275
  unique: 24
  examples: ['Excellent', 'Excellent', 'Excellent', 'Excellent', 'Excellent']

Donor_Age
  dtype: float64
  non-null: 1050
  unique: 57
  examples: ['57.0', '57.0', '51.0', '51.0', '55.0']

Tissue_Serial
  dtype: int64
  non-null: 1055
  unique: 1054
  examples: ['12346', '12654', '12655', '12656', '12657']

Donor_Sex
  dtype: str
  non-null: 566
  unique: 22
  examples: ['Male', 'Male', 'Female', 'Female', 'Female']

Donor_Graft_Size_mm
  dtype: object
  non-null: 234
  unique: 29
  examples: ['8.25', '8', '8.5',

In [9]:
id_audit = labelled_df[["Tissue_ID"]].copy()

id_audit["Tissue_ID"] = id_audit["Tissue_ID"].astype("string").str.strip()

print("Unique tissue IDs:", id_audit["Tissue_ID"].nunique())
print("Duplicate tissue IDs:", id_audit["Tissue_ID"].duplicated().sum())

display(
    id_audit["Tissue_ID"]
    .str.extract(r"^(.*?)(?:\s+[RL])?$", expand=False)
    .value_counts()
    .head(20)
)

Unique tissue IDs: 1036
Duplicate tissue IDs: 0


Tissue_ID
NEB-2020-09-12654    1
NEB-2020-09-12655    1
NEB-2020-09-12656    1
NEB-2020-09-12657    1
NEB-2020-09-12658    1
NEB-2020-09-12659    1
NEB-2020-09-12660    1
NEB-2020-09-12661    1
NEB-2020-09-12662    1
NEB-2020-09-12663    1
NEB-2020-09-12664    1
NEB-2020-09-12665    1
NEB-2020-09-12666    1
NEB-2020-09-12667    1
NEB-2020-09-12668    1
NEB-2020-09-12669    1
NEB-2020-09-12670    1
NEB-2020-09-12671    1
NEB-2020-09-12672    1
NEB-2020-09-12673    1
Name: count, dtype: Int64

In [10]:
for col in [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Requested_Grade",
    "Grade_Request_Met",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Review_Notes",
    "Additional_Findings",
]:
    if col in labelled_df.columns:
        print(f"\n--- {col} ---")
        display(labelled_df[col].value_counts(dropna=False).head(15))


--- Overall_Donor_Quality ---


Overall_Donor_Quality
NaN                                                                762
Excellent                                                          136
Very good                                                           79
Good                                                                31
Very good (as read)                                                  4
Excellent (as read)                                                  2
Good (as read)                                                       2
Good (Very good struck)                                              2
Excellent / very good (mark spans both - verify)                     2
Very good (Good also marked - verify)                                1
Very good (stroke runs across Excellent and Very good - verify)      1
Very good (as read - marks also near Excellent/Good)                 1
Excellent / Very good (stroke crosses both - verify)                 1
Good (as read - tick trails from Very good)            


--- Transplant_Suitability ---


Transplant_Suitability
NaN                                                                                                          582
Suitable (released for surgery - printed release form)                                                       197
Suitable                                                                                                     175
Not stated (no release form in PDF); donor sheet grade A+                                                      9
Not ticked on Donor Form B                                                                                     9
Released (printed release form); Donor Form B suitability unticked                                             7
Suitable (Donor Form B; released for surgery - printed release form)                                           7
Suitable (released for surgery - printed release form); Donor Form B stroke crosses suitable/not suitable      5
Not stated (no release form in PDF); donor sheet grade A                 


--- Requested_Grade ---


...

Tissue_Outcome
NaN                                                                                      986
Exported - cornea sent to Dr Y.Y. Choong, Malaysia, 23/09/2025 (handwritten note)          3
Allocated to Dr Pattaramon, Rutnin Medical, Thailand (29/09/2020)                          2
Cornea expired on 03.01.2025 (handwritten)                                                 2
Exported - cornea sent to KPJ Eye Specialist, Malaysia, 26/12/2024 (handwritten note)      2
Cornea expired 16/10/2025 (handwritten note)                                               2
Cornea expired on 23.10.2025 (handwritten)                                                 2
Cornea expired on 28.10.2025 (handwritten)                                                 2
Cornea expired on 31/10/2025 (handwritten)                                                 2
Cornea expired 11/11/2025 (handwritten note)                                               2
Discarded - Hep C positive (handwritten note)          


--- Review_Notes ---


Review_Notes
NaN                                                                                                                                             749
Folds call uncertain                                                                                                                             43
Guttata call uncertain                                                                                                                           16
CD 2538 within 50 of the 2,500 A/A+ cut-off                                                                                                      10
Guttata call uncertain; Guttata label (reference: ~50% A at CD ≥ 2,500)                                                                           7
Guttata label (reference: ~50% A at CD ≥ 2,500)                                                                                                   7
CD in 2200–2300 borderline zone; CD 2268 within 50 of the 2,250 B/A cut-off                        


--- Additional_Findings ---


Additional_Findings
NaN                                                                                                                                                         464
Rim 3-4; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                          60
Rim 3-3.5; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                        16
Rim 2.5-4; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                        10
Rim 3-5; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                          10
Rim 2-3; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                           9
Rim 2.5-3.5; cornea 

In [11]:
serial_col = "Tissue_Serial"

print("Rows:", len(labelled_df))
print("Unique serials:", labelled_df[serial_col].nunique())
print("Duplicate serial rows:", labelled_df[serial_col].duplicated(keep=False).sum())

duplicate_serials = labelled_df[
    labelled_df[serial_col].duplicated(keep=False)
].sort_values(serial_col)

display(duplicate_serials[[
    "Tissue_ID", "Tissue_Serial", "Eye",
    "Image_ID", TARGET
]])

Rows: 1036
Unique serials: 1036
Duplicate serial rows: 0


,Tissue_ID,Tissue_Serial,Eye,Image_ID,Tissue_Grade_Clean


In [12]:
# Look for left/right records sharing the same serial or ID prefix.
id_check = labelled_df[["Tissue_ID", "Tissue_Serial", "Eye", TARGET]].copy()

id_check["Tissue_ID"] = id_check["Tissue_ID"].astype("string").str.strip()

id_check["ID_without_eye_suffix"] = (
    id_check["Tissue_ID"]
    .str.replace(r"\s+[RL](?:\s+\(.*\))?$", "", regex=True)
)

prefix_counts = (
    id_check.groupby("ID_without_eye_suffix", dropna=False)
    .agg(
        record_count=("Tissue_ID", "size"),
        unique_eyes=("Eye", "nunique"),
        unique_tissue_ids=("Tissue_ID", "nunique"),
    )
    .sort_values("record_count", ascending=False)
)

display(prefix_counts.head(20))

,record_count,unique_eyes,unique_tissue_ids
ID_without_eye_suffix,,,
NEB-2020-09-12654,1,1,1
NEB-2020-09-12655,1,1,1
NEB-2020-09-12656,1,1,1
NEB-2020-09-12657,1,1,1
NEB-2020-09-12658,1,1,1
NEB-2020-09-12659,1,1,1
NEB-2020-09-12660,1,1,1
NEB-2020-09-12661,1,1,1
NEB-2020-09-12662,1,1,1


In [13]:
grouping_candidates = [
    "Tissue_ID",
    "Tissue_Serial",
    "Image_ID",
    "Donor_Age",
    "Donor_Sex",
]

for col in grouping_candidates:
    if col in labelled_df.columns:
        print(
            f"{col}: {labelled_df[col].nunique(dropna=True)} unique, "
            f"{labelled_df[col].isna().sum()} missing"
        )

Tissue_ID: 1036 unique, 0 missing
Tissue_Serial: 1036 unique, 0 missing
Image_ID: 999 unique, 37 missing
Donor_Age: 57 unique, 2 missing
Donor_Sex: 21 unique, 478 missing


In [14]:
# 1. Inspect duplicated tissue serials
duplicate_serials = labelled_df[
    labelled_df["Tissue_Serial"].duplicated(keep=False)
].sort_values("Tissue_Serial")

print("Duplicated serial records:")
display(
    duplicate_serials[
        ["Tissue_ID", "Tissue_Serial", "Eye", "Image_ID", TARGET]
    ]
)

# 2. Check whether removing the eye suffix reveals repeated IDs
id_check = labelled_df[["Tissue_ID", "Eye", TARGET]].copy()

id_check["ID_without_eye_suffix"] = (
    id_check["Tissue_ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+[RL](?:\s+\(.*\))?$", "", regex=True)
)

repeated_prefixes = (
    id_check.groupby("ID_without_eye_suffix")
    .agg(
        records=("Tissue_ID", "size"),
        eyes=("Eye", "nunique"),
    )
    .query("records > 1")
    .sort_values("records", ascending=False)
)

print("Repeated ID prefixes:", len(repeated_prefixes))
display(repeated_prefixes.head(20))

Duplicated serial records:


,Tissue_ID,Tissue_Serial,Eye,Image_ID,Tissue_Grade_Clean


Repeated ID prefixes: 0


,records,eyes
ID_without_eye_suffix,,


In [15]:

# Check whether any available columns can identify the same donor
# across multiple records.

grouping_candidates = [
    "Tissue_ID",
    "Tissue_Serial",
    "Image_ID",
    "Donor_Age",
    "Donor_Sex",
]

for col in grouping_candidates:
    if col in labelled_df.columns:
        print(f"\n--- {col} ---")
        print("Non-null:", labelled_df[col].notna().sum())
        print("Unique:", labelled_df[col].nunique(dropna=True))
        print("Repeated values:", labelled_df[col].duplicated(keep=False).sum())

print("\nPotential donor-identification columns:")
print([
    col for col in labelled_df.columns
    if "donor" in col.lower()
    and any(term in col.lower() for term in ["id", "number", "code"])
])



--- Tissue_ID ---
Non-null: 1036
Unique: 1036
Repeated values: 0

--- Tissue_Serial ---
Non-null: 1036
Unique: 1036
Repeated values: 0

--- Image_ID ---
Non-null: 999
Unique: 999
Repeated values: 37

--- Donor_Age ---
Non-null: 1034
Unique: 57
Repeated values: 1029

--- Donor_Sex ---
Non-null: 558
Unique: 21
Repeated values: 1020

Potential donor-identification columns:
[]


In [16]:

# Define candidate feature groups for controlled model comparisons.

numeric_features = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Donor_Age",
]

clinical_features = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
    "Epi_Has_Exposure",
    "Epi_Exposure_Severity",
    "Epi_Exposure_Location",
    "Epi_Has_Defect",
    "Epi_Has_Haze",
    "Epi_Haze_Severity",
    "Epi_Has_Arcus",
]

# ECD will be evaluated separately as a baseline and as a feature.
ecd_feature = ["ECD_CD_cells_per_mm2"]

numeric_without_ecd = [
    col for col in numeric_features if col not in ecd_feature
]

# Confirm all candidate features exist.
for group_name, features in {
    "Numeric": numeric_features,
    "Clinical": clinical_features,
    "Numeric without ECD": numeric_without_ecd,
}.items():
    missing = sorted(set(features) - set(labelled_df.columns))
    print(f"{group_name}: {len(features)} features")
    print("Missing columns:", missing)


Numeric: 7 features
Missing columns: []
Clinical: 24 features
Missing columns: []
Numeric without ECD: 6 features
Missing columns: []


In [19]:

# Audit candidate feature types and missingness.

candidate_features = numeric_features + clinical_features

feature_audit = pd.DataFrame({
    "Feature": candidate_features,
    "Data_Type": [
        str(labelled_df[col].dtype)
        for col in candidate_features
    ],
    "Non_Missing": [
        labelled_df[col].notna().sum()
        for col in candidate_features
    ],
    "Missing": [
        labelled_df[col].isna().sum()
        for col in candidate_features
    ],
    "Missing_Percent": [
        round(labelled_df[col].isna().mean() * 100, 2)
        for col in candidate_features
    ],
    "Unique_Values": [
        labelled_df[col].nunique(dropna=True)
        for col in candidate_features
    ],
})

print(display(
    feature_audit.sort_values(
        ["Missing_Percent", "Feature"],
        ascending=[False, True]
    ).reset_index(drop=True)
))


,Feature,Data_Type,Non_Missing,Missing,Missing_Percent,Unique_Values
0,Epi_Haze_Severity,str,29,1007,97.20,2
1,Endo_RBC_Severity,str,30,1006,97.10,3
2,Stroma_Scar_Type,str,33,1003,96.81,2
3,Endo_Cell_Dropout_Severity,str,91,945,91.22,2
4,Epi_Exposure_Location,str,307,729,70.37,4
5,Epi_Exposure_Severity,str,307,729,70.37,3
6,Stroma_Arcus_Severity,str,475,561,54.15,3
7,Endo_Stress_Line_Amount,str,488,548,52.90,3
8,Endo_Stress_Line_Location,str,501,535,51.64,4
9,Descemet_Fold_Relative_Number,str,529,507,48.94,5


None


In [20]:

# Inspect the recorded values of clinical features.
# This does not modify the dataset.

clinical_value_summary = []

for col in clinical_features:
    values = labelled_df[col].dropna()

    clinical_value_summary.append({
        "Feature": col,
        "Data_Type": str(labelled_df[col].dtype),
        "Recorded": len(values),
        "Unique_Values": values.nunique(),
        "Examples": values.astype(str).unique()[:10].tolist(),
    })

clinical_value_summary = pd.DataFrame(clinical_value_summary)

display(
    clinical_value_summary.sort_values(
        ["Recorded", "Feature"],
        ascending=[True, True]
    ).reset_index(drop=True)
)


,Feature,Data_Type,Recorded,Unique_Values,Examples
0,Epi_Haze_Severity,str,29,2,"[Unspecified, Mild]"
1,Endo_RBC_Severity,str,30,3,"[Unspecified, ++, +]"
2,Stroma_Scar_Type,str,33,2,"[Other scar, Phaco scar]"
3,Endo_Cell_Dropout_Severity,str,91,2,"[Mild, Moderate]"
4,Epi_Exposure_Location,str,307,4,"[Peripheral, Unspecified, Central, Central + P..."
5,Epi_Exposure_Severity,str,307,3,"[Mild, Unspecified, Very mild]"
6,Stroma_Arcus_Severity,str,475,3,"[Unspecified, Mild, Moderate]"
7,Endo_Stress_Line_Amount,str,488,3,"[Few, Unspecified, Several]"
8,Endo_Stress_Line_Location,str,501,4,"[Central, Peripheral, Unspecified, Central + P..."
9,Descemet_Fold_Relative_Number,str,529,5,"[Unspecified, Several, Mild, Moderate, Several..."


In [21]:

# Inspect binary clinical values and their missingness.
# This cell does not modify the dataset.

binary_clinical_features = [
    col for col in clinical_features
    if labelled_df[col].dropna().isin([0, 1, 0.0, 1.0]).all()
    and labelled_df[col].notna().any()
]

binary_summary = []

for col in binary_clinical_features:
    counts = labelled_df[col].value_counts(dropna=False)

    binary_summary.append({
        "Feature": col,
        "Present (1)": int((labelled_df[col] == 1).sum()),
        "Absent (0)": int((labelled_df[col] == 0).sum()),
        "Missing": int(labelled_df[col].isna().sum()),
    })

display(
    pd.DataFrame(binary_summary)
    .sort_values("Feature")
    .reset_index(drop=True)
)


,Feature,Present (1),Absent (0),Missing
0,Descemet_Has_Folds,523,6,507
1,Endo_Has_Cell_Dropout,91,470,475
2,Endo_Has_RBC,30,531,475
3,Endo_Has_Stress_Lines,488,73,475
4,Epi_Has_Arcus,326,246,464
5,Epi_Has_Defect,4,568,464
6,Epi_Has_Exposure,307,265,464
7,Epi_Has_Haze,29,543,464
8,Stroma_Has_Arcus,475,98,463
9,Stroma_Has_Edema,10,563,463


In [22]:
# Print the recorded categories for clinical fields.
# This cell does not modify the dataset.

categorical_clinical_features = [
    col for col in clinical_features
    if labelled_df[col].dtype == "str"
]

for col in categorical_clinical_features:
    print(f"\n--- {col} ---")
    print("Missing:", labelled_df[col].isna().sum())
    print(labelled_df[col].value_counts(dropna=False).to_string())


--- Stroma_Arcus_Severity ---
Missing: 561
Stroma_Arcus_Severity
NaN            561
Unspecified    432
Mild            38
Moderate         5

--- Stroma_Scar_Type ---
Missing: 1003
Stroma_Scar_Type
NaN           1003
Other scar      27
Phaco scar       6

--- Descemet_Fold_Severity ---
Missing: 507
Descemet_Fold_Severity
NaN            507
Mild           384
Unspecified    107
Moderate        21
Very mild       17

--- Descemet_Fold_Relative_Number ---
Missing: 507
Descemet_Fold_Relative_Number
NaN                 507
Unspecified         249
Mild                249
Several              28
Moderate              2
Several/Moderate      1

--- Endo_Stress_Line_Location ---
Missing: 535
Endo_Stress_Line_Location
NaN                     535
Central                 237
Unspecified             174
Peripheral               63
Central + Peripheral     27

--- Endo_Stress_Line_Amount ---
Missing: 548
Endo_Stress_Line_Amount
NaN            548
Few            468
Unspecified     18
Several       

In [23]:
# Check consistency between clinical flags and their related details.
# This cell does not modify the dataset.

clinical_pairs = {
    "Descemet_Has_Folds": [
        "Descemet_Fold_Severity",
        "Descemet_Fold_Relative_Number",
    ],
    "Endo_Has_Stress_Lines": [
        "Endo_Stress_Line_Location",
        "Endo_Stress_Line_Amount",
    ],
    "Endo_Has_Cell_Dropout": [
        "Endo_Cell_Dropout_Severity",
    ],
    "Endo_Has_RBC": [
        "Endo_RBC_Severity",
    ],
    "Epi_Has_Exposure": [
        "Epi_Exposure_Severity",
        "Epi_Exposure_Location",
    ],
    "Epi_Has_Haze": [
        "Epi_Haze_Severity",
    ],
    "Stroma_Has_Arcus": [
        "Stroma_Arcus_Severity",
    ],
    "Stroma_Has_Scar": [
        "Stroma_Scar_Type",
    ],
}

for flag, detail_columns in clinical_pairs.items():
    print(f"\n--- {flag} ---")

    for detail in detail_columns:
        recorded_detail = labelled_df[detail].notna()
        flag_missing = labelled_df[flag].isna()
        flag_zero = labelled_df[flag].eq(0)
        flag_one = labelled_df[flag].eq(1)

        print(f"\nDetail: {detail}")
        print("Detail recorded:", int(recorded_detail.sum()))
        print("Detail recorded while flag = 0:",
              int((recorded_detail & flag_zero).sum()))
        print("Detail recorded while flag = 1:",
              int((recorded_detail & flag_one).sum()))
        print("Detail recorded while flag is missing:",
              int((recorded_detail & flag_missing).sum()))


--- Descemet_Has_Folds ---

Detail: Descemet_Fold_Severity
Detail recorded: 529
Detail recorded while flag = 0: 6
Detail recorded while flag = 1: 523
Detail recorded while flag is missing: 0

Detail: Descemet_Fold_Relative_Number
Detail recorded: 529
Detail recorded while flag = 0: 6
Detail recorded while flag = 1: 523
Detail recorded while flag is missing: 0

--- Endo_Has_Stress_Lines ---

Detail: Endo_Stress_Line_Location
Detail recorded: 501
Detail recorded while flag = 0: 13
Detail recorded while flag = 1: 488
Detail recorded while flag is missing: 0

Detail: Endo_Stress_Line_Amount
Detail recorded: 488
Detail recorded while flag = 0: 0
Detail recorded while flag = 1: 488
Detail recorded while flag is missing: 0

--- Endo_Has_Cell_Dropout ---

Detail: Endo_Cell_Dropout_Severity
Detail recorded: 91
Detail recorded while flag = 0: 0
Detail recorded while flag = 1: 91
Detail recorded while flag is missing: 0

--- Endo_Has_RBC ---

Detail: Endo_RBC_Severity
Detail recorded: 30
Detail 

In [24]:

# Inspect clinical detail fields recorded when the binary flag is 0.
# This cell does not modify the dataset.

checks = {
    "Descemet_Has_Folds": [
        "Descemet_Fold_Severity",
        "Descemet_Fold_Relative_Number",
    ],
    "Endo_Has_Stress_Lines": [
        "Endo_Stress_Line_Location",
        "Endo_Stress_Line_Amount",
    ],
}

for flag, detail_columns in checks.items():
    mask = labelled_df[flag].eq(0) & labelled_df[detail_columns].notna().any(axis=1)

    print(f"\n{'=' * 60}")
    print(f"Flag: {flag}")
    print("Rows to inspect:", int(mask.sum()))

    if mask.any():
        columns_to_show = [
            "Tissue_ID",
            "Image_ID",
            TARGET,
            flag,
            *detail_columns,
        ]

        print(labelled_df.loc[mask, columns_to_show].to_string(index=False))
    else:
        print("No potential inconsistencies found.")



Flag: Descemet_Has_Folds
Rows to inspect: 6
          Tissue_ID               Image_ID Tissue_Grade_Clean  Descemet_Has_Folds Descemet_Fold_Severity Descemet_Fold_Relative_Number
NEB-2022-04-13824 R NEB-2022-04-13824 R_SP                  B                 0.0            Unspecified                   Unspecified
NEB-2022-10-14330 L NEB-2022-10-14330 L_SP                 A+                 0.0            Unspecified                   Unspecified
NEB-2022-10-14336 R NEB-2022-10-14336 R_SP                 A+                 0.0            Unspecified                   Unspecified
NEB-2024-12-16603 R NEB-2024-12-16603 R_SP                  A                 0.0                   Mild                          Mild
NEB-2025-11-17623 R NEB-2025-11-17623 R_SP                  A                 0.0            Unspecified                   Unspecified
NEB-2025-11-17624 L NEB-2025-11-17624 L_SP                  A                 0.0            Unspecified                   Unspecified

Flag: End

In [25]:

# Summarize potential conflicts between binary flags and clinical details.
# This cell does not modify the dataset.

review_cases = [
    (
        "Descemet_Has_Folds",
        ["Descemet_Fold_Severity", "Descemet_Fold_Relative_Number"],
    ),
    (
        "Endo_Has_Stress_Lines",
        ["Endo_Stress_Line_Location", "Endo_Stress_Line_Amount"],
    ),
]

for flag, details in review_cases:
    mask = labelled_df[flag].eq(0) & labelled_df[details].notna().any(axis=1)
    subset = labelled_df.loc[mask, [flag] + details]

    print(f"\n--- {flag}: {len(subset)} potential conflicts ---")

    for column in details:
        print(f"\n{column} values:")
        print(subset[column].value_counts(dropna=False).to_string())

    print("\nCombined detail patterns:")
    print(
        subset[details]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )



--- Descemet_Has_Folds: 6 potential conflicts ---

Descemet_Fold_Severity values:
Descemet_Fold_Severity
Unspecified    5
Mild           1

Descemet_Fold_Relative_Number values:
Descemet_Fold_Relative_Number
Unspecified    5
Mild           1

Combined detail patterns:
Descemet_Fold_Severity  Descemet_Fold_Relative_Number
Unspecified             Unspecified                      5
Mild                    Mild                             1

--- Endo_Has_Stress_Lines: 13 potential conflicts ---

Endo_Stress_Line_Location values:
Endo_Stress_Line_Location
Unspecified             6
Central                 6
Central + Peripheral    1

Endo_Stress_Line_Amount values:
Endo_Stress_Line_Amount
NaN    13

Combined detail patterns:
Endo_Stress_Line_Location  Endo_Stress_Line_Amount
Unspecified                <MISSING>                  6
Central                    <MISSING>                  6
Central + Peripheral       <MISSING>                  1


In [26]:

# Review clinical flag counts by specialist-reviewed tissue grade.
# Read-only: no changes to the dataset.

flags_to_review = [
    "Descemet_Has_Folds",
    "Endo_Has_Stress_Lines",
    "Endo_Has_Cell_Dropout",
    "Endo_Has_RBC",
    "Epi_Has_Exposure",
    "Epi_Has_Haze",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
]

for flag in flags_to_review:
    print(f"\n--- {flag} by {TARGET} ---")

    counts = pd.crosstab(
        labelled_df[TARGET],
        labelled_df[flag].fillna(-1).map({
            -1: "Missing",
            0: "Absent",
            1: "Present",
        }),
        dropna=False,
    )

    print(counts.to_string())



--- Descemet_Has_Folds by Tissue_Grade_Clean ---
Descemet_Has_Folds  Absent  Missing  Present
Tissue_Grade_Clean                          
A                        3      137      179
A+                       2      275      268
B                        1       95       76

--- Endo_Has_Stress_Lines by Tissue_Grade_Clean ---
Endo_Has_Stress_Lines  Absent  Missing  Present
Tissue_Grade_Clean                             
A                          34      120      165
A+                         26      262      257
B                          13       93       66

--- Endo_Has_Cell_Dropout by Tissue_Grade_Clean ---
Endo_Has_Cell_Dropout  Absent  Missing  Present
Tissue_Grade_Clean                             
A                         165      120       34
A+                        240      262       43
B                          65       93       14

--- Endo_Has_RBC by Tissue_Grade_Clean ---
Endo_Has_RBC        Absent  Missing  Present
Tissue_Grade_Clean                          
A    

In [27]:

# Summarize clinical feature coverage.
# Read-only: this cell does not modify the dataset.

coverage_rows = []

for feature in clinical_features:
    recorded = labelled_df[feature].notna().sum()
    missing = labelled_df[feature].isna().sum()

    coverage_rows.append({
        "Feature": feature,
        "Recorded": int(recorded),
        "Missing": int(missing),
        "Coverage_pct": round(recorded / len(labelled_df) * 100, 2),
        "Unique_recorded_values": int(labelled_df[feature].nunique()),
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df = coverage_df.sort_values(
    by="Coverage_pct",
    ascending=False,
)

print("Total labelled rows:", len(labelled_df))
print("\nClinical feature coverage:")
print(coverage_df.to_string(index=False))


Total labelled rows: 1036

Clinical feature coverage:
                      Feature  Recorded  Missing  Coverage_pct  Unique_recorded_values
              Stroma_Is_Clear       573      463         55.31                       2
            Stroma_Is_Compact       573      463         55.31                       2
             Stroma_Has_Arcus       573      463         55.31                       2
             Stroma_Has_Edema       573      463         55.31                       2
              Stroma_Has_Scar       573      463         55.31                       2
                Epi_Has_Arcus       572      464         55.21                       2
                 Epi_Has_Haze       572      464         55.21                       2
               Epi_Has_Defect       572      464         55.21                       2
             Epi_Has_Exposure       572      464         55.21                       2
        Endo_Has_Stress_Lines       561      475         54.15              

In [28]:

# Proposed clinical feature groups for initial experiments.
# These lists do not modify labelled_df.

clinical_binary_candidates = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Epi_Has_Arcus",
    "Epi_Has_Defect",
    "Epi_Has_Exposure",
    "Epi_Has_Haze",
    "Endo_Has_Stress_Lines",
    "Endo_Has_RBC",
    "Endo_Has_Cell_Dropout",
    "Descemet_Has_Folds",
]

clinical_categorical_candidates = [
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Stroma_Arcus_Severity",
    "Epi_Exposure_Location",
    "Epi_Exposure_Severity",
]

clinical_sparse_review = [
    "Endo_Cell_Dropout_Severity",
    "Stroma_Scar_Type",
    "Endo_RBC_Severity",
    "Epi_Haze_Severity",
]

print("Binary clinical candidates:", len(clinical_binary_candidates))
print(*clinical_binary_candidates, sep="\n  ")

print("\nCategorical clinical candidates:", len(clinical_categorical_candidates))
print(*clinical_categorical_candidates, sep="\n  ")

print("\nSparse features for later review:", len(clinical_sparse_review))
print(*clinical_sparse_review, sep="\n  ")

all_selected = (
    clinical_binary_candidates
    + clinical_categorical_candidates
    + clinical_sparse_review
)

missing_columns = [
    column for column in all_selected
    if column not in labelled_df.columns
]

print("\nColumns not found in dataset:", missing_columns)
print("Unique selected columns:", len(set(all_selected)))


Binary clinical candidates: 13
Stroma_Is_Clear
  Stroma_Is_Compact
  Stroma_Has_Arcus
  Stroma_Has_Edema
  Stroma_Has_Scar
  Epi_Has_Arcus
  Epi_Has_Defect
  Epi_Has_Exposure
  Epi_Has_Haze
  Endo_Has_Stress_Lines
  Endo_Has_RBC
  Endo_Has_Cell_Dropout
  Descemet_Has_Folds

Categorical clinical candidates: 7
Descemet_Fold_Severity
  Descemet_Fold_Relative_Number
  Endo_Stress_Line_Location
  Endo_Stress_Line_Amount
  Stroma_Arcus_Severity
  Epi_Exposure_Location
  Epi_Exposure_Severity

Sparse features for later review: 4
Endo_Cell_Dropout_Severity
  Stroma_Scar_Type
  Endo_RBC_Severity
  Epi_Haze_Severity

Columns not found in dataset: []
Unique selected columns: 24


In [29]:

# Audit values before preprocessing.
# Read-only: this cell does not modify labelled_df.

feature_groups = {
    "Binary clinical candidates": clinical_binary_candidates,
    "Categorical clinical candidates": clinical_categorical_candidates,
    "Sparse features for later review": clinical_sparse_review,
}

for group_name, features in feature_groups.items():
    print(f"\n{'=' * 70}")
    print(group_name)
    print("=" * 70)

    for feature in features:
        values = labelled_df[feature].value_counts(dropna=False)

        print(f"\n{feature}")
        print("Distinct values including missing:", len(values))
        print(values.to_string())



Binary clinical candidates

Stroma_Is_Clear
Distinct values including missing: 3
Stroma_Is_Clear
1.0    564
NaN    463
0.0      9

Stroma_Is_Compact
Distinct values including missing: 3
Stroma_Is_Compact
NaN    463
0.0    344
1.0    229

Stroma_Has_Arcus
Distinct values including missing: 3
Stroma_Has_Arcus
1.0    475
NaN    463
0.0     98

Stroma_Has_Edema
Distinct values including missing: 3
Stroma_Has_Edema
0.0    563
NaN    463
1.0     10

Stroma_Has_Scar
Distinct values including missing: 3
Stroma_Has_Scar
0.0    540
NaN    463
1.0     33

Epi_Has_Arcus
Distinct values including missing: 3
Epi_Has_Arcus
NaN    464
1.0    326
0.0    246

Epi_Has_Defect
Distinct values including missing: 3
Epi_Has_Defect
0.0    568
NaN    464
1.0      4

Epi_Has_Exposure
Distinct values including missing: 3
Epi_Has_Exposure
NaN    464
1.0    307
0.0    265

Epi_Has_Haze
Distinct values including missing: 3
Epi_Has_Haze
0.0    543
NaN    464
1.0     29

Endo_Has_Stress_Lines
Distinct values includin

In [30]:

# Validate values in binary clinical candidates.
# Read-only: no changes to labelled_df.

for feature in clinical_binary_candidates:
    print(f"\n--- {feature} ---")
    print(
        labelled_df[feature]
        .value_counts(dropna=False)
        .to_string()
    )

    unexpected = labelled_df.loc[
        labelled_df[feature].notna()
        & ~labelled_df[feature].isin([0, 1]),
        feature,
    ]

    print("Unexpected non-binary values:", len(unexpected))
    if not unexpected.empty:
        print(unexpected.value_counts().to_string())



--- Stroma_Is_Clear ---
Stroma_Is_Clear
1.0    564
NaN    463
0.0      9
Unexpected non-binary values: 0

--- Stroma_Is_Compact ---
Stroma_Is_Compact
NaN    463
0.0    344
1.0    229
Unexpected non-binary values: 0

--- Stroma_Has_Arcus ---
Stroma_Has_Arcus
1.0    475
NaN    463
0.0     98
Unexpected non-binary values: 0

--- Stroma_Has_Edema ---
Stroma_Has_Edema
0.0    563
NaN    463
1.0     10
Unexpected non-binary values: 0

--- Stroma_Has_Scar ---
Stroma_Has_Scar
0.0    540
NaN    463
1.0     33
Unexpected non-binary values: 0

--- Epi_Has_Arcus ---
Epi_Has_Arcus
NaN    464
1.0    326
0.0    246
Unexpected non-binary values: 0

--- Epi_Has_Defect ---
Epi_Has_Defect
0.0    568
NaN    464
1.0      4
Unexpected non-binary values: 0

--- Epi_Has_Exposure ---
Epi_Has_Exposure
NaN    464
1.0    307
0.0    265
Unexpected non-binary values: 0

--- Epi_Has_Haze ---
Epi_Has_Haze
0.0    543
NaN    464
1.0     29
Unexpected non-binary values: 0

--- Endo_Has_Stress_Lines ---
Endo_Has_Stress_L

In [31]:

# Validate categories for the clinical categorical candidates.
# Read-only: no changes to labelled_df.

for feature in clinical_categorical_candidates:
    print(f"\n--- {feature} ---")

    counts = labelled_df[feature].value_counts(dropna=False)
    print(counts.to_string())

    recorded = int(labelled_df[feature].notna().sum())
    missing = int(labelled_df[feature].isna().sum())

    print("Recorded:", recorded)
    print("Missing:", missing)
    print("Distinct non-missing categories:",
          int(labelled_df[feature].nunique(dropna=True)))



--- Descemet_Fold_Severity ---
Descemet_Fold_Severity
NaN            507
Mild           384
Unspecified    107
Moderate        21
Very mild       17
Recorded: 529
Missing: 507
Distinct non-missing categories: 4

--- Descemet_Fold_Relative_Number ---
Descemet_Fold_Relative_Number
NaN                 507
Unspecified         249
Mild                249
Several              28
Moderate              2
Several/Moderate      1
Recorded: 529
Missing: 507
Distinct non-missing categories: 5

--- Endo_Stress_Line_Location ---
Endo_Stress_Line_Location
NaN                     535
Central                 237
Unspecified             174
Peripheral               63
Central + Peripheral     27
Recorded: 501
Missing: 535
Distinct non-missing categories: 4

--- Endo_Stress_Line_Amount ---
Endo_Stress_Line_Amount
NaN            548
Few            468
Unspecified     18
Several          2
Recorded: 488
Missing: 548
Distinct non-missing categories: 3

--- Stroma_Arcus_Severity ---
Stroma_Arcus_Severity
Na

In [32]:

# Define numeric feature groups for the initial experiments.
# This cell only defines lists and checks their columns.

ecd_features = [
    "ECD_CD_cells_per_mm2",
]

morphology_features = [
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

numeric_with_ecd = ecd_features + morphology_features

numeric_with_age = numeric_with_ecd + ["Donor_Age"]

feature_sets = {
    "ECD-only baseline": ecd_features,
    "Morphology excluding ECD": morphology_features,
    "ECD + morphology": numeric_with_ecd,
    "ECD + morphology + donor age": numeric_with_age,
}

for name, features in feature_sets.items():
    missing_columns = [
        feature for feature in features
        if feature not in labelled_df.columns
    ]

    print(f"\n--- {name} ---")
    print("Features:", features)
    print("Feature count:", len(features))
    print("Missing columns:", missing_columns)

print("\nTarget distribution:")
print(labelled_df[TARGET].value_counts().to_string())



--- ECD-only baseline ---
Features: ['ECD_CD_cells_per_mm2']
Feature count: 1
Missing columns: []

--- Morphology excluding ECD ---
Features: ['CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Feature count: 5
Missing columns: []

--- ECD + morphology ---
Features: ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Feature count: 6
Missing columns: []

--- ECD + morphology + donor age ---
Features: ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2', 'Donor_Age']
Feature count: 7
Missing columns: []

Target distribution:
Tissue_Grade_Clean
A+    545
A     319
B     172


In [33]:

# Define a reusable numeric preprocessing pipeline.
# No model is trained in this cell.

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

print("Numeric preprocessing pipeline:")
print(numeric_preprocessor)

print("\nMissing-value counts by numeric feature set:")

for name, features in feature_sets.items():
    missing_counts = labelled_df[features].isna().sum()

    print(f"\n--- {name} ---")
    print(missing_counts.to_string())
    print("Rows with at least one missing feature:",
          int(labelled_df[features].isna().any(axis=1).sum()))
    print("Rows complete across all features:",
          int(labelled_df[features].notna().all(axis=1).sum()))

print("\nTarget values:")
print(sorted(labelled_df[TARGET].unique().tolist()))


Numeric preprocessing pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

Missing-value counts by numeric feature set:

--- ECD-only baseline ---
ECD_CD_cells_per_mm2    96
Rows with at least one missing feature: 96
Rows complete across all features: 940

--- Morphology excluding ECD ---
CV                  96
HEX_percent         96
SD                  96
Total_Cell_Count    96
Total_Area_um2      96
Rows with at least one missing feature: 96
Rows complete across all features: 940

--- ECD + morphology ---
ECD_CD_cells_per_mm2    96
CV                      96
HEX_percent             96
SD                      96
Total_Cell_Count        96
Total_Area_um2          96
Rows with at least one missing feature: 96
Rows complete across all features: 940

--- ECD + morphology + donor age ---
ECD_CD_cells_per_mm2    96
CV                      96
HEX_percent             96
SD                      96
Total_Cell_Count        96
Total_Area_um2          96
Donor_Age            

In [34]:

# Define the clinical preprocessing pipeline.
# No model is trained and labelled_df is not modified.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

clinical_binary_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ]
)

clinical_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            clinical_binary_preprocessor,
            clinical_binary_candidates,
        ),
        (
            "categorical",
            clinical_categorical_preprocessor,
            clinical_categorical_candidates,
        ),
    ],
    remainder="drop",
)

print("Clinical preprocessing pipeline:")
print(clinical_preprocessor)

print("\nBinary input features:", len(clinical_binary_candidates))
print("Categorical input features:", len(clinical_categorical_candidates))
print(
    "Total input features:",
    len(clinical_binary_candidates)
    + len(clinical_categorical_candidates),
)

print("\nSparse features excluded from this initial pipeline:")
print(*clinical_sparse_review, sep="\n")


Clinical preprocessing pipeline:
ColumnTransformer(transformers=[('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median'))]),
                                 ['Stroma_Is_Clear', 'Stroma_Is_Compact',
                                  'Stroma_Has_Arcus', 'Stroma_Has_Edema',
                                  'Stroma_Has_Scar', 'Epi_Has_Arcus',
                                  'Epi_Has_Defect', 'Epi_Has_Exposure',
                                  'Epi_Has_Haze', 'Endo_Has_Stress_Lines',
                                  'Endo_Has_RBC', 'Endo_Has_Cell_Dropout',
                                  'Descemet_Has_Folds']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_freque

In [35]:

# Refine clinical preprocessing.
# Binary flags remain 0/1 after imputation.
# No model is trained and labelled_df is not modified.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

clinical_binary_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            ),
        ),
    ]
)

clinical_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            clinical_binary_preprocessor,
            clinical_binary_candidates,
        ),
        (
            "categorical",
            clinical_categorical_preprocessor,
            clinical_categorical_candidates,
        ),
    ],
    remainder="drop",
)

print("Updated clinical preprocessing pipeline:")
print(clinical_preprocessor)
print("\nBinary features:", len(clinical_binary_candidates))
print("Categorical features:", len(clinical_categorical_candidates))


Updated clinical preprocessing pipeline:
ColumnTransformer(transformers=[('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='most_frequent'))]),
                                 ['Stroma_Is_Clear', 'Stroma_Is_Compact',
                                  'Stroma_Has_Arcus', 'Stroma_Has_Edema',
                                  'Stroma_Has_Scar', 'Epi_Has_Arcus',
                                  'Epi_Has_Defect', 'Epi_Has_Exposure',
                                  'Epi_Has_Haze', 'Endo_Has_Stress_Lines',
                                  'Endo_Has_RBC', 'Endo_Has_Cell_Dropout',
                                  'Descemet_Has_Folds']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strate

In [36]:

# Diagnostic check of transformed clinical features.
# This is not model evaluation.

X_clinical = labelled_df[
    clinical_binary_candidates + clinical_categorical_candidates
]

X_clinical_transformed = clinical_preprocessor.fit_transform(X_clinical)

print("Original clinical matrix shape:", X_clinical.shape)
print("Transformed clinical matrix shape:", X_clinical_transformed.shape)
print(
    "Transformed matrix contains NaN:",
    bool(pd.isna(X_clinical_transformed).any()),
)
print("Target rows:", len(labelled_df))
print("Target classes:", sorted(labelled_df[TARGET].unique().tolist()))


Original clinical matrix shape: (1036, 20)
Transformed clinical matrix shape: (1036, 52)
Transformed matrix contains NaN: False
Target rows: 1036
Target classes: ['A', 'A+', 'B']


In [37]:

# Combined numeric + clinical preprocessing.
# This defines the transformer; it does not train a model.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

combined_numeric_features = numeric_with_ecd
combined_binary_features = clinical_binary_candidates
combined_categorical_features = clinical_categorical_candidates

combined_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median", add_indicator=True),
            combined_numeric_features,
        ),
        (
            "clinical_binary",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            ),
            combined_binary_features,
        ),
        (
            "clinical_categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent"),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(handle_unknown="ignore"),
                    ),
                ]
            ),
            combined_categorical_features,
        ),
    ],
    remainder="drop",
)

print("Combined preprocessing pipeline:")
print(combined_preprocessor)

print("\nNumeric features:", len(combined_numeric_features))
print("Binary clinical features:", len(combined_binary_features))
print("Categorical clinical features:", len(combined_categorical_features))
print(
    "Total original input columns:",
    len(combined_numeric_features)
    + len(combined_binary_features)
    + len(combined_categorical_features),
)


Combined preprocessing pipeline:
ColumnTransformer(transformers=[('numeric',
                                 SimpleImputer(add_indicator=True,
                                               strategy='median'),
                                 ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent',
                                  'SD', 'Total_Cell_Count', 'Total_Area_um2']),
                                ('clinical_binary',
                                 SimpleImputer(add_indicator=True,
                                               strategy='most_frequent'),
                                 ['Stroma_Is_Clear', 'Stroma_Is_Compact',
                                  'Stroma_Has_Arcus', 'Stroma_Has_Edema',
                                  'Stroma_Has_Sc...
                                  'Descemet_Has_Folds']),
                                ('clinical_categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImput

In [38]:

# Diagnostic check of the combined preprocessing pipeline.
# This is not model evaluation.

combined_features = (
    combined_numeric_features
    + combined_binary_features
    + combined_categorical_features
)

X_combined = labelled_df[combined_features]
X_combined_transformed = combined_preprocessor.fit_transform(X_combined)

print("Original combined matrix shape:", X_combined.shape)
print("Transformed combined matrix shape:", X_combined_transformed.shape)
print(
    "Transformed matrix contains NaN:",
    bool(pd.isna(X_combined_transformed).any()),
)
print("Target rows:", len(labelled_df))
print("Target classes:", sorted(labelled_df[TARGET].unique().tolist()))


Original combined matrix shape: (1036, 26)
Transformed combined matrix shape: (1036, 64)
Transformed matrix contains NaN: False
Target rows: 1036
Target classes: ['A', 'A+', 'B']


In [39]:

# Check the modelling environment before defining evaluation splits.

import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split

print("scikit-learn version:", sklearn.__version__)
print("Training target size:", len(labelled_df))
print("\nTarget distribution:")
print(labelled_df[TARGET].value_counts().to_string())

print("\nTarget proportions:")
print(
    (labelled_df[TARGET].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

print("\nStratifiedKFold available:", StratifiedKFold is not None)
print("train_test_split available:", train_test_split is not None)


scikit-learn version: 1.9.1
Training target size: 1036

Target distribution:
Tissue_Grade_Clean
A+    545
A     319
B     172

Target proportions:
Tissue_Grade_Clean
A+    52.61
A     30.79
B     16.60

StratifiedKFold available: True
train_test_split available: True


In [40]:

# Create a reproducible stratified train/test split.
# No model or preprocessing transformer is fitted here.

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

indices = labelled_df.index

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=labelled_df[TARGET],
)

y_train = labelled_df.loc[train_indices, TARGET]
y_test = labelled_df.loc[test_indices, TARGET]

print("Random state:", RANDOM_STATE)
print("Training rows:", len(train_indices))
print("Test rows:", len(test_indices))

print("\nTraining grade counts:")
print(y_train.value_counts().to_string())

print("\nTest grade counts:")
print(y_test.value_counts().to_string())

print("\nTraining grade proportions (%):")
print((y_train.value_counts(normalize=True) * 100).round(2).to_string())

print("\nTest grade proportions (%):")
print((y_test.value_counts(normalize=True) * 100).round(2).to_string())

print("\nIndex overlap between train and test:",
      len(set(train_indices) & set(test_indices)))


Random state: 42
Training rows: 828
Test rows: 208

Training grade counts:
Tissue_Grade_Clean
A+    436
A     255
B     137

Test grade counts:
Tissue_Grade_Clean
A+    109
A      64
B      35

Training grade proportions (%):
Tissue_Grade_Clean
A+    52.66
A     30.80
B     16.55

Test grade proportions (%):
Tissue_Grade_Clean
A+    52.40
A     30.77
B     16.83

Index overlap between train and test: 0


In [41]:

# ECD-only baseline: preprocessing is fitted on training data only.

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline

ecd_baseline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

X_train_ecd = labelled_df.loc[train_indices, ecd_features]
X_test_ecd = labelled_df.loc[test_indices, ecd_features]

ecd_baseline.fit(X_train_ecd, y_train)
y_pred_ecd = ecd_baseline.predict(X_test_ecd)

print("ECD-only baseline results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_ecd), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred_ecd), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_ecd, average="macro"), 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_ecd,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("Confusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(confusion_matrix(y_test, y_pred_ecd, labels=["A+", "A", "B"]))


ECD-only baseline results
Accuracy: 0.6538
Balanced accuracy: 0.7002
Macro F1: 0.6816

Classification report:
              precision    recall  f1-score   support

          A+     0.8101    0.5872    0.6809       109
           A     0.4615    0.6562    0.5419        64
           B     0.7895    0.8571    0.8219        35

    accuracy                         0.6538       208
   macro avg     0.6870    0.7002    0.6816       208
weighted avg     0.6994    0.6538    0.6618       208

Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[64 45  0]
 [14 42  8]
 [ 1  4 30]]


In [42]:

# Morphology-only baseline: deliberately excludes ECD.
# Imputation is fitted on training data only.

morphology_baseline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

X_train_morph = labelled_df.loc[train_indices, morphology_features]
X_test_morph = labelled_df.loc[test_indices, morphology_features]

morphology_baseline.fit(X_train_morph, y_train)
y_pred_morph = morphology_baseline.predict(X_test_morph)

print("Morphology-only baseline results")
print("Features:", morphology_features)
print("Accuracy:", round(accuracy_score(y_test, y_pred_morph), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, y_pred_morph), 4),
)
print("Macro F1:", round(f1_score(y_test, y_pred_morph, average="macro"), 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_morph,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(confusion_matrix(y_test, y_pred_morph, labels=["A+", "A", "B"]))


Morphology-only baseline results
Features: ['CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Accuracy: 0.726
Balanced accuracy: 0.7589
Macro F1: 0.7515

Classification report:
              precision    recall  f1-score   support

          A+     0.8765    0.6514    0.7474       109
           A     0.5368    0.7969    0.6415        64
           B     0.9062    0.8286    0.8657        35

    accuracy                         0.7260       208
   macro avg     0.7732    0.7589    0.7515       208
weighted avg     0.7770    0.7260    0.7347       208


Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[71 38  0]
 [10 51  3]
 [ 0  6 29]]


C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [43]:

# Re-evaluate ECD-only and morphology-only baselines with scaling.
# Preprocessing is fitted on training data only.

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_numeric_baseline():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

baseline_results = {}

baseline_feature_sets = {
    "ECD only": ecd_features,
    "Morphology only": morphology_features,
}

for name, features in baseline_feature_sets.items():
    X_train = labelled_df.loc[train_indices, features]
    X_test = labelled_df.loc[test_indices, features]

    model = make_numeric_baseline()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    baseline_results[name] = {
        "model": model,
        "predictions": predictions,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_f1": f1_score(
            y_test,
            predictions,
            average="macro",
        ),
    }

    print(f"\n--- {name} ---")
    print("Accuracy:", round(baseline_results[name]["accuracy"], 4))
    print(
        "Balanced accuracy:",
        round(baseline_results[name]["balanced_accuracy"], 4),
    )
    print("Macro F1:", round(baseline_results[name]["macro_f1"], 4))
    print(
        "Iterations used:",
        model.named_steps["classifier"].n_iter_.tolist(),
    )

print("\n--- Comparison ---")
for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
    print(f"\n{metric}:")
    for name in baseline_results:
        print(
            f"  {name}: "
            f"{baseline_results[name][metric]:.4f}"
        )



--- ECD only ---
Accuracy: 0.6538
Balanced accuracy: 0.7002
Macro F1: 0.6816
Iterations used: [9]

--- Morphology only ---
Accuracy: 0.7404
Balanced accuracy: 0.7724
Macro F1: 0.7665
Iterations used: [26]

--- Comparison ---

accuracy:
  ECD only: 0.6538
  Morphology only: 0.7404

balanced_accuracy:
  ECD only: 0.7002
  Morphology only: 0.7724

macro_f1:
  ECD only: 0.6816
  Morphology only: 0.7665


In [44]:

# ECD + morphology baseline.
# Imputation and scaling are fitted on training data only.

combined_numeric_baseline = make_numeric_baseline()

X_train_numeric = labelled_df.loc[train_indices, numeric_with_ecd]
X_test_numeric = labelled_df.loc[test_indices, numeric_with_ecd]

combined_numeric_baseline.fit(X_train_numeric, y_train)
y_pred_numeric = combined_numeric_baseline.predict(X_test_numeric)

combined_numeric_results = {
    "accuracy": accuracy_score(y_test, y_pred_numeric),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred_numeric),
    "macro_f1": f1_score(
        y_test,
        y_pred_numeric,
        average="macro",
    ),
}

print("ECD + morphology results")
for metric, value in combined_numeric_results.items():
    print(f"{metric}: {value:.4f}")

print(
    "Iterations used:",
    combined_numeric_baseline.named_steps["classifier"].n_iter_.tolist(),
)

print("\nComparison with individual baselines:")
for name, results in baseline_results.items():
    print(f"\n{name}")
    print("Accuracy:", round(results["accuracy"], 4))
    print("Balanced accuracy:", round(results["balanced_accuracy"], 4))
    print("Macro F1:", round(results["macro_f1"], 4))

print("\nECD + morphology")
for metric, value in combined_numeric_results.items():
    print(f"{metric}: {value:.4f}")


ECD + morphology results
accuracy: 0.7404
balanced_accuracy: 0.7724
macro_f1: 0.7665
Iterations used: [23]

Comparison with individual baselines:

ECD only
Accuracy: 0.6538
Balanced accuracy: 0.7002
Macro F1: 0.6816

Morphology only
Accuracy: 0.7404
Balanced accuracy: 0.7724
Macro F1: 0.7665

ECD + morphology
accuracy: 0.7404
balanced_accuracy: 0.7724
macro_f1: 0.7665


In [45]:

# Compare predictions from morphology-only and ECD + morphology.
# Uses the same test rows for both fitted models.

morph_predictions = baseline_results["Morphology only"]["predictions"]
combined_predictions = y_pred_numeric

prediction_comparison = pd.DataFrame({
    "Actual": y_test,
    "Morphology_Prediction": morph_predictions,
    "ECD_Morphology_Prediction": combined_predictions,
})

changed = (
    prediction_comparison["Morphology_Prediction"]
    != prediction_comparison["ECD_Morphology_Prediction"]
)

print("Test rows:", len(prediction_comparison))
print("Predictions changed after adding ECD:", int(changed.sum()))
print(
    "Predictions unchanged:",
    int((~changed).sum()),
)

if changed.any():
    print("\nRows where predictions differ:")
    print(prediction_comparison.loc[changed].to_string())

print("\nCorrect predictions:")
print(
    "Morphology only:",
    int(
        (
            prediction_comparison["Morphology_Prediction"]
            == prediction_comparison["Actual"]
        ).sum(),
    ),
)
print(
    "ECD + morphology:",
    int(
        (
            prediction_comparison["ECD_Morphology_Prediction"]
            == prediction_comparison["Actual"]
        ).sum(),
    ),
)


Test rows: 208
Predictions changed after adding ECD: 0
Predictions unchanged: 208

Correct predictions:
Morphology only: 154
ECD + morphology: 154


## Preliminary Numeric Baseline Results

Three scaled logistic regression baselines were evaluated on the same stratified holdout test set of 208 samples. Median imputation and standardization were fitted using training data only, and class weights were balanced.

| Model | Accuracy | Balanced Accuracy | Macro F1 |
|---|---:|---:|---:|
| ECD only | 0.6538 | 0.7002 | 0.6816 |
| Morphology excluding ECD | 0.7404 | 0.7724 | 0.7665 |
| ECD + morphology | 0.7404 | 0.7724 | 0.7665 |

The morphology-only model outperformed the ECD-only baseline across all three metrics. Adding ECD to the morphology-only model did not change any of the 208 test predictions.

These findings are preliminary and apply to one row-level holdout split. Donor-independent generalization has not yet been established because a reliable donor identifier is unavailable. Further validation and experiments with clinical features are required before drawing final conclusions.